In [1]:
import pandas as pd
import xgboost as xgb

In [3]:
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

In [2]:
url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "car_fuel_efficiency_2026.csv"
)

df = pd.read_csv(url).fillna(0)

print("Dimensions :", df.shape)
df.head()

Dimensions : (10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

print("Train :", len(df_train))
print("Validation :", len(df_val))
print("Test :", len(df_test))

Train : 6000
Validation : 2000
Test : 2000


In [5]:
target = "fuel_efficiency_mpg"

y_train = df_train[target].to_numpy()
y_val = df_val[target].to_numpy()

train_dicts = df_train.drop(columns=target).to_dict(orient="records")
val_dicts = df_val.drop(columns=target).to_dict(orient="records")

In [6]:
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

feature_names = dv.get_feature_names_out()

print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print(feature_names)

Train : (6000, 16)
Validation : (2000, 16)
['acceleration' 'drivetrain=All-wheel drive'
 'drivetrain=Front-wheel drive' 'drivetrain=Rear-wheel drive'
 'engine_displacement' 'fuel_type=Diesel' 'fuel_type=Gasoline'
 'fuel_type=Hybrid' 'horsepower' 'model_year' 'num_cylinders' 'num_doors'
 'origin=Asia' 'origin=Europe' 'origin=USA' 'vehicle_weight']


In [7]:
tree = DecisionTreeRegressor(
    max_depth=1,
    random_state=1,
)

tree.fit(X_train, y_train)

print(export_text(tree, feature_names=list(feature_names)))

split_feature = feature_names[tree.tree_.feature[0]]
original_feature = split_feature.split("=")[0]

print("Réponse Q1 :", original_feature)

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]

Réponse Q1 : model_year


In [8]:
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1,
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
score_q2 = rmse(y_val, y_pred)

print(f"Validation RMSE : {score_q2:.6f}")

Validation RMSE : 1.833220


In [9]:
results_q3 = []

for n in [10, 50, 100, 150]:
    model = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    score = rmse(y_val, model.predict(X_val))

    results_q3.append({
        "n_estimators": n,
        "rmse": score,
        "rmse_3_decimals": round(score, 3),
    })

results_q3 = pd.DataFrame(results_q3)
results_q3

,n_estimators,rmse,rmse_3_decimals
0,10,1.833220,1.833
1,50,1.773988,1.774
2,100,1.769786,1.770
3,150,1.769198,1.769


In [10]:
best_index = results_q3["rmse_3_decimals"].idxmin()
best_n = int(results_q3.loc[best_index, "n_estimators"])

print("Réponse Q3 :", best_n)

Réponse Q3 : 150


In [11]:
results_q4 = []

for depth in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        model = RandomForestRegressor(
            max_depth=depth,
            n_estimators=n,
            random_state=1,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)
        score = rmse(y_val, model.predict(X_val))

        results_q4.append({
            "max_depth": depth,
            "n_estimators": n,
            "rmse": score,
        })

results_q4 = pd.DataFrame(results_q4)

results_q4.pivot(
    index="max_depth",
    columns="n_estimators",
    values="rmse",
)

n_estimators,10,50,100,150
max_depth,,,,
10,1.782348,1.747520,1.743986,1.743716
15,1.831390,1.770883,1.767872,1.766752
20,1.837314,1.772377,1.768918,1.768476
25,1.836852,1.772123,1.768320,1.767927


In [12]:
mean_rmse = results_q4.groupby("max_depth")["rmse"].mean()

display(mean_rmse.to_frame(name="mean_rmse"))

best_depth = int(mean_rmse.idxmin())
print("Réponse Q4 :", best_depth)

,mean_rmse
max_depth,
10,1.754392
15,1.784224
20,1.786771
25,1.786305


Réponse Q4 : 10


In [13]:
model_q5 = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1,
)

model_q5.fit(X_train, y_train)

importance = pd.Series(
    model_q5.feature_importances_,
    index=feature_names,
    name="importance",
)

importance.sort_values(ascending=False).to_frame()

,importance
model_year,0.296107
vehicle_weight,0.203313
fuel_type=Gasoline,0.160712
horsepower,0.078271
engine_displacement,0.061281
acceleration,0.054998
origin=USA,0.042152
drivetrain=Front-wheel drive,0.020062
num_doors,0.019401
fuel_type=Diesel,0.016571


In [14]:
candidates = [
    "vehicle_weight",
    "horsepower",
    "acceleration",
    "engine_displacement",
]

candidate_importance = importance.loc[candidates].sort_values(
    ascending=False
)

display(candidate_importance.to_frame())
print("Réponse Q5 :", candidate_importance.idxmax())

,importance
vehicle_weight,0.203313
horsepower,0.078271
engine_displacement,0.061281
acceleration,0.054998


Réponse Q5 : vehicle_weight


In [15]:
dtrain = xgb.DMatrix(
    X_train,
    label=y_train,
    feature_names=list(feature_names),
)

dval = xgb.DMatrix(
    X_val,
    label=y_val,
    feature_names=list(feature_names),
)

watchlist = [
    (dtrain, "train"),
    (dval, "val"),
]

In [16]:
results_q6 = []
histories = {}

for eta in [0.3, 0.1]:
    params = {
        "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "nthread": 8,
        "seed": 1,
        "verbosity": 1,
    }

    history = {}

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100,
        evals=watchlist,
        evals_result=history,
        verbose_eval=20,
    )

    histories[eta] = history

    results_q6.append({
        "eta": eta,
        "validation_rmse": rmse(y_val, model.predict(dval)),
    })

results_q6 = pd.DataFrame(results_q6)
results_q6

[0]	train-rmse:2.40653	val-rmse:2.46782
[20]	train-rmse:1.39368	val-rmse:1.73766
[40]	train-rmse:1.25326	val-rmse:1.76296
[60]	train-rmse:1.10265	val-rmse:1.79401
[80]	train-rmse:0.99487	val-rmse:1.81308
[99]	train-rmse:0.90106	val-rmse:1.82646
[0]	train-rmse:2.74739	val-rmse:2.78635
[20]	train-rmse:1.55847	val-rmse:1.75076
[40]	train-rmse:1.43074	val-rmse:1.71289
[60]	train-rmse:1.37884	val-rmse:1.71688
[80]	train-rmse:1.34578	val-rmse:1.72100
[99]	train-rmse:1.31192	val-rmse:1.72481


,eta,validation_rmse
0,0.3,1.826458
1,0.1,1.724810


In [17]:
best_index = results_q6["validation_rmse"].idxmin()
best_eta = results_q6.loc[best_index, "eta"]

print("Réponse Q6 :", best_eta)

Réponse Q6 : 0.1
